# QA/QC: Pipeline Output Inspection

Scan `output/` for CSV and NetCDF files produced by the incremental pipeline.

- **Topography** (CSV): plot each variable as a choropleth
- **Land cover** (CSV): extract majority NLCD class per HRU and plot
- **Climate** (NetCDF): plot the last time-step for each variable

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

In [ ]:
OUTPUT_DIR = Path("../output")
FABRIC_PATH = Path("../data/pywatershed_gis/drb_2yr/nhru.gpkg")

fabric = gpd.read_file(FABRIC_PATH)
print(f"Fabric: {len(fabric)} features, CRS={fabric.crs}")

# NLCD class labels for majority-class plots
NLCD_LABELS = {
    11: "Open Water",
    12: "Perennial Ice/Snow",
    21: "Developed, Open",
    22: "Developed, Low",
    23: "Developed, Med",
    24: "Developed, High",
    31: "Barren Land",
    41: "Deciduous Forest",
    42: "Evergreen Forest",
    43: "Mixed Forest",
    51: "Dwarf Scrub",
    52: "Shrub/Scrub",
    71: "Grassland/Herbaceous",
    72: "Sedge/Herbaceous",
    73: "Lichens",
    74: "Moss",
    81: "Pasture/Hay",
    82: "Cultivated Crops",
    90: "Woody Wetlands",
    95: "Emergent Herbaceous Wetlands",
}

In [ ]:
csv_files = sorted(OUTPUT_DIR.rglob("*.csv"))
nc_files = sorted(OUTPUT_DIR.rglob("*.nc"))

print(f"Found {len(csv_files)} CSV files:")
for f in csv_files:
    print(f"  {f.relative_to(OUTPUT_DIR)}")
print(f"\nFound {len(nc_files)} NetCDF files:")
for f in nc_files:
    print(f"  {f.relative_to(OUTPUT_DIR)}")

In [ ]:
def join_series_to_fabric(fabric, nhm_ids, values):
    """Join a series of values keyed by nhm_id to the fabric GeoDataFrame."""
    lookup = dict(zip(nhm_ids, values, strict=False))
    gdf = fabric.copy()
    gdf["_val"] = gdf["nhm_id"].map(lookup)
    return gdf.dropna(subset=["_val"])


def plot_choropleth(gdf, title, ax, categorical=False, cmap="viridis", legend_kwds=None):
    """Plot a choropleth map on the given axes."""
    if gdf.empty or gdf["_val"].isna().all():
        ax.set_title(f"{title}\n(no data)")
        ax.set_axis_off()
        return
    if legend_kwds is None:
        legend_kwds = {"shrink": 0.6} if not categorical else {"loc": "lower left", "fontsize": 6}
    gdf.plot(
        column="_val",
        ax=ax,
        legend=True,
        cmap=cmap,
        categorical=categorical,
        legend_kwds=legend_kwds,
    )
    ax.set_title(title, fontsize=10)
    ax.set_axis_off()

## Topography (static CSVs)

Each CSV has columns `nhm_id, <variable>`. Plot as continuous choropleths.

In [ ]:
topo_dir = OUTPUT_DIR / "topography"
topo_files = sorted(topo_dir.glob("*.csv")) if topo_dir.exists() else []

if topo_files:
    ncols = min(3, len(topo_files))
    nrows = (len(topo_files) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
    axes = np.atleast_1d(axes).ravel()

    for i, csv_path in enumerate(topo_files):
        df = pd.read_csv(csv_path)
        var_name = [c for c in df.columns if c != "nhm_id"][0]
        gdf = join_series_to_fabric(fabric, df["nhm_id"], df[var_name])
        plot_choropleth(gdf, var_name, axes[i])

    for j in range(len(topo_files), len(axes)):
        axes[j].set_visible(False)
    fig.suptitle("Topography", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("No topography CSVs found.")

## Land Cover (categorical CSVs)

Each CSV has columns `nhm_id, <prefix>_<class_code>, ..., <prefix>_count`.
Class columns hold fractional area. Extract the majority (highest-fraction) class per HRU.

In [ ]:
lc_dir = OUTPUT_DIR / "land_cover"
lc_files = sorted(lc_dir.glob("*.csv")) if lc_dir.exists() else []

if lc_files:
    ncols = min(2, len(lc_files))
    nrows = (len(lc_files) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(8 * ncols, 6 * nrows))
    axes = np.atleast_1d(axes).ravel()

    for i, csv_path in enumerate(lc_files):
        df = pd.read_csv(csv_path)
        # Identify class fraction columns (exclude nhm_id and _count)
        class_cols = [c for c in df.columns if c != "nhm_id" and not c.endswith("_count")]
        # Extract NLCD class code from column name (last segment after '_')
        class_codes = [int(c.rsplit("_", 1)[-1]) for c in class_cols]

        # Majority class = column with highest fraction per row
        fractions = df[class_cols].values
        majority_idx = np.argmax(fractions, axis=1)
        majority_codes = np.array(class_codes)[majority_idx]

        # Map codes to labels for legend
        majority_labels = [NLCD_LABELS.get(c, str(c)) for c in majority_codes]
        gdf = join_series_to_fabric(fabric, df["nhm_id"], majority_labels)

        title = csv_path.stem + " (majority class)"
        plot_choropleth(gdf, title, axes[i], categorical=True, cmap="tab20")

    for j in range(len(lc_files), len(axes)):
        axes[j].set_visible(False)
    fig.suptitle("Land Cover — Majority Class", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("No land cover CSVs found.")

## Climate (temporal NetCDF)

Each NetCDF has dimensions `(time, nhm_id)`. Plot the last time-step for each variable.

In [ ]:
climate_dir = OUTPUT_DIR / "climate"
climate_files = sorted(climate_dir.glob("*.nc")) if climate_dir.exists() else []

for nc_path in climate_files:
    ds = xr.open_dataset(nc_path)
    last_time = pd.Timestamp(ds.time.values[-1]).strftime("%Y-%m-%d")
    ds_last = ds.isel(time=-1)

    # ID dimension
    id_dim = "nhm_id" if "nhm_id" in ds.dims else list(ds.dims - {"time"})[0]
    data_vars = [v for v in ds_last.data_vars if id_dim in ds_last[v].dims]

    ncols = min(3, len(data_vars))
    nrows = (len(data_vars) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
    axes = np.atleast_1d(axes).ravel()

    for i, var_name in enumerate(data_vars):
        da = ds_last[var_name]
        gdf = join_series_to_fabric(fabric, da[id_dim].values, da.values)
        plot_choropleth(gdf, var_name, axes[i])

    for j in range(len(data_vars), len(axes)):
        axes[j].set_visible(False)

    rel_path = nc_path.relative_to(OUTPUT_DIR)
    fig.suptitle(f"{rel_path} (last step: {last_time})", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()
    ds.close()

if not climate_files:
    print("No climate NetCDF files found.")